# 2048 — DQN Baseline

Reproduces the DQN baseline from *2048: Reinforcement Learning in a Delayed
Reward Environment* (Saligram, Bhathal, Manihani — Stanford CS224R 2025):
log2-flattened board state, reward = merge score + monotonicity bonus,
epsilon-greedy over legal moves, Huber loss with a Polyak-averaged target
network.

The agent code itself is **not** duplicated here — this notebook installs it
straight from the GitHub repo (`rl2048` package) so the notebook and the
repo never drift apart. To iterate on the agent, edit the `.py` files in
[the repo](https://github.com/fourofclubs001/autoencoder_cumple_juli),
push, then re-run the install cell below to pick up the change.

Enable a GPU accelerator for this kernel (Settings → Accelerator) before
running — the network itself is tiny, but the wall-clock savings mostly
come from a faster CPU/less contention on Kaggle vs. a laptop.

In [ ]:
# Install the rl2048 package (and its gymnasium-2048 dependency) directly
# from the GitHub repo. Re-run this cell after pushing changes to the repo.
GITHUB_REPO = "https://github.com/fourofclubs001/autoencoder_cumple_juli.git"

import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"git+{GITHUB_REPO}"],
    check=True,
)


In [ ]:
import time
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from rl2048.dqn_agent import DQNAgent
from rl2048.env import TwentyFortyEightWrapper

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## Config

In [ ]:
EPISODES = 5000
MAX_STEPS = 10_000
EPS_START = 1.0
EPS_END = 0.05
LOG_EVERY = 20
CHECKPOINT_EVERY = 500

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUT_DIR / "dqn_2048.pt"
HISTORY_PATH = OUT_DIR / "dqn_2048_history.csv"


## Train

Logs per-episode score and max tile to a DataFrame (and checkpoints
periodically to `OUT_DIR`) so a Kaggle session timeout doesn't lose
everything — re-running this cell resumes from the last checkpoint.

In [ ]:
def epsilon_by_episode(episode, total_episodes, eps_start, eps_end):
    decay_rate = -np.log(eps_end / eps_start) / total_episodes
    return max(eps_end, eps_start * np.exp(-decay_rate * episode))


env = TwentyFortyEightWrapper()
agent = DQNAgent()

start_episode = 1
history = []

if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=agent.device)
    agent.online.load_state_dict(ckpt["online"])
    agent.target.load_state_dict(ckpt["target"])
    agent.optimizer.load_state_dict(ckpt["optimizer"])
    start_episode = ckpt["episode"] + 1
    history = pd.read_csv(HISTORY_PATH).to_dict("records") if HISTORY_PATH.exists() else []
    print(f"Resumed from episode {ckpt['episode']}")

scores = deque(maxlen=100)
max_tiles = deque(maxlen=100)

pbar = tqdm(range(start_episode, EPISODES + 1))
for episode in pbar:
    state, _ = env.reset()
    legal_mask = env.legal_action_mask()
    epsilon = epsilon_by_episode(episode, EPISODES, EPS_START, EPS_END)

    info = {}
    for _ in range(MAX_STEPS):
        action = agent.act(state, legal_mask, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        next_legal_mask = env.legal_action_mask()
        done = terminated or truncated

        agent.remember(state, action, reward, next_state, done, next_legal_mask)
        agent.learn()

        state = next_state
        legal_mask = next_legal_mask
        if done:
            break

    scores.append(info["total_score"])
    max_tiles.append(2 ** info["max"])
    history.append({
        "episode": episode,
        "score": info["total_score"],
        "max_tile": 2 ** info["max"],
        "epsilon": epsilon,
    })

    if episode % LOG_EVERY == 0:
        pbar.set_description(
            f"avg_score(100)={np.mean(scores):7.1f} avg_max_tile(100)={np.mean(max_tiles):6.1f}"
        )

    if episode % CHECKPOINT_EVERY == 0 or episode == EPISODES:
        torch.save({
            "episode": episode,
            "online": agent.online.state_dict(),
            "target": agent.target.state_dict(),
            "optimizer": agent.optimizer.state_dict(),
        }, CHECKPOINT_PATH)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

print("Done.")


## Results

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(HISTORY_PATH)
df["avg_score_100"] = df["score"].rolling(100, min_periods=1).mean()
df["avg_max_tile_100"] = df["max_tile"].rolling(100, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df["episode"], df["score"], alpha=0.3, color="tab:orange")
axes[0].plot(df["episode"], df["avg_score_100"], color="tab:red")
axes[0].set_title("Score per episode")
axes[0].set_xlabel("Episode")

axes[1].plot(df["episode"], df["max_tile"], alpha=0.3, color="tab:orange")
axes[1].plot(df["episode"], df["avg_max_tile_100"], color="tab:red")
axes[1].set_title("Max tile per episode")
axes[1].set_xlabel("Episode")
axes[1].set_yscale("log", base=2)

plt.tight_layout()
plt.show()

print(f"Max score: {df['score'].max()}")
print(f"Max tile:  {int(df['max_tile'].max())}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")
print(f"History saved to:    {HISTORY_PATH}")
